# Leafspy — NZ Nissan Leaf Market Dashboard

Interactive analysis of TradeMe Nissan Leaf listings. All plots below respond to the shared filter widgets in cell 2.

Run `python fetch_leafs.py` first to populate `leafs.csv`.

In [ ]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, Markdown

from leafspy.schema import TRIM_GRADES, BATTERY_STATUSES, BODY_STATUSES, PRICE_TYPES
from leafspy.summarize import summarize, SellerCar

df = pd.read_csv('leafs.csv')
print(f'Loaded {len(df)} listings, {df["effective_price"].notna().sum()} with usable price.')
df.head()

## Shared filters
All plots below re-render when these change.

In [ ]:
trim_w = widgets.SelectMultiple(
    options=list(TRIM_GRADES),
    value=list(TRIM_GRADES),
    description='Trim:',
    rows=len(TRIM_GRADES),
)
battery_kwh_w = widgets.SelectMultiple(
    options=[24, 30, 40, 62, 0],   # 0 represents "unknown" (None in data)
    value=[24, 30, 40, 62, 0],
    description='kWh:',
    rows=5,
)
battery_status_w = widgets.SelectMultiple(
    options=list(BATTERY_STATUSES),
    value=list(BATTERY_STATUSES),
    description='Battery:',
    rows=len(BATTERY_STATUSES),
)
body_status_w = widgets.SelectMultiple(
    options=list(BODY_STATUSES),
    value=list(BODY_STATUSES),
    description='Body:',
    rows=len(BODY_STATUSES),
)
price_type_w = widgets.SelectMultiple(
    options=list(PRICE_TYPES),
    value=list(PRICE_TYPES),
    description='Price type:',
    rows=len(PRICE_TYPES),
)
year_min = int(df['year'].min()) if df['year'].notna().any() else 2010
year_max = int(df['year'].max()) if df['year'].notna().any() else 2025
year_w = widgets.IntRangeSlider(
    value=[year_min, year_max], min=year_min, max=year_max, step=1,
    description='Year:', continuous_update=False,
)
x_axis_w = widgets.Dropdown(
    options=['year', 'odometer', 'soh_percent'],
    value='year',
    description='X-axis:',
)

controls = widgets.VBox([
    widgets.HBox([trim_w, battery_kwh_w]),
    widgets.HBox([battery_status_w, body_status_w]),
    widgets.HBox([price_type_w]),
    year_w,
    x_axis_w,
])
display(controls)

def apply_filters(df, trims, kwhs, batt, body, ptypes, year_range):
    # Map 0 back to NaN for unknown kWh
    kwh_set = set(kwhs)
    include_unknown_kwh = 0 in kwh_set
    kwh_set.discard(0)

    mask = (
        df['trim_grade'].isin(trims)
        & df['battery_status'].isin(batt)
        & df['body_status'].isin(body)
        & df['year'].between(year_range[0], year_range[1])
    )
    if ptypes:
        mask = mask & (df['price_type'].isin(ptypes) | df['price_type'].isna())
    if include_unknown_kwh:
        kwh_mask = df['battery_kwh'].isin(kwh_set) | df['battery_kwh'].isna()
    else:
        kwh_mask = df['battery_kwh'].isin(kwh_set)
    return df[mask & kwh_mask]

## 2D scatter — price vs (year | odometer | SOH)
Color: trim grade. Hover for listing details.

In [ ]:
def plot_scatter(trims, kwhs, batt, body, ptypes, year_range, x_axis):
    sub = apply_filters(df, trims, kwhs, batt, body, ptypes, year_range)
    sub = sub[sub['effective_price'].notna() & sub[x_axis].notna()]
    if len(sub) == 0:
        print('No data matches filters.')
        return
    hover_cols = ['title', 'region', 'year', 'odometer', 'soh_percent', 'battery_status', 'body_status', 'listing_url']
    fig = px.scatter(
        sub, x=x_axis, y='effective_price', color='trim_grade',
        symbol='battery_status',
        hover_data={c: True for c in hover_cols if c in sub.columns},
        title=f'Effective price vs {x_axis} (n={len(sub)})',
        height=550,
    )
    fig.show()

out_scatter = widgets.interactive_output(plot_scatter, {
    'trims': trim_w, 'kwhs': battery_kwh_w, 'batt': battery_status_w,
    'body': body_status_w, 'ptypes': price_type_w, 'year_range': year_w,
    'x_axis': x_axis_w,
})
display(out_scatter)

## 3D scatter — year × SOH × price
Drag to rotate. Color: trim grade.

In [ ]:
def plot_3d(trims, kwhs, batt, body, ptypes, year_range):
    sub = apply_filters(df, trims, kwhs, batt, body, ptypes, year_range)
    sub = sub[sub['effective_price'].notna() & sub['year'].notna() & sub['soh_percent'].notna()]
    if len(sub) == 0:
        print('No data with all three axes available. Many listings will be missing SOH.')
        return
    fig = px.scatter_3d(
        sub, x='year', y='soh_percent', z='effective_price', color='trim_grade',
        hover_data=['title', 'region', 'listing_url'],
        title=f'year × SOH × price (n={len(sub)})',
        height=650,
    )
    fig.show()

out_3d = widgets.interactive_output(plot_3d, {
    'trims': trim_w, 'kwhs': battery_kwh_w, 'batt': battery_status_w,
    'body': body_status_w, 'ptypes': price_type_w, 'year_range': year_w,
})
display(out_3d)

## Battery × Body condition 2×2 box plot
Surfaces the "battery donor" and "shell car" segments.

In [ ]:
def plot_2x2(trims, kwhs, batt, body, ptypes, year_range):
    sub = apply_filters(df, trims, kwhs, batt, body, ptypes, year_range)
    sub = sub[sub['effective_price'].notna()]
    if len(sub) == 0:
        print('No data matches filters.')
        return
    sub = sub.assign(cell=sub['battery_status'] + ' / ' + sub['body_status'])
    fig = px.box(
        sub, x='cell', y='effective_price',
        color='battery_status',
        points='all', hover_data=['title', 'listing_url'],
        title=f'Price by battery × body condition (n={len(sub)})',
        height=550,
    )
    fig.update_xaxes(tickangle=-30)
    fig.show()

out_2x2 = widgets.interactive_output(plot_2x2, {
    'trims': trim_w, 'kwhs': battery_kwh_w, 'batt': battery_status_w,
    'body': body_status_w, 'ptypes': price_type_w, 'year_range': year_w,
})
display(out_2x2)

## Trim premium box plot — faceted by battery status
Whether the 30G premium holds across damaged tiers.

In [ ]:
def plot_trim_box(trims, kwhs, batt, body, ptypes, year_range):
    sub = apply_filters(df, trims, kwhs, batt, body, ptypes, year_range)
    sub = sub[sub['effective_price'].notna()]
    if len(sub) == 0:
        print('No data matches filters.')
        return
    fig = px.box(
        sub, x='trim_grade', y='effective_price',
        facet_col='battery_status',
        points='all', hover_data=['title', 'listing_url'],
        title=f'Price by trim_grade, faceted by battery_status (n={len(sub)})',
        height=500,
    )
    fig.show()

out_trim = widgets.interactive_output(plot_trim_box, {
    'trims': trim_w, 'kwhs': battery_kwh_w, 'batt': battery_status_w,
    'body': body_status_w, 'ptypes': price_type_w, 'year_range': year_w,
})
display(out_trim)

## Closest-comparable summary
Edit the `SellerCar(...)` to match your car's actual spec.

In [ ]:
my_car = SellerCar(
    trim_grade='G',
    battery_kwh=30,
    year=2016,
    battery_status='failed',   # battery is shot
    body_status='roadworthy',  # otherwise fine
)
display(Markdown(summarize(df, my_car, year_tolerance=2)))